# 01 Vegetation — Summer NDVI, Change & Dated Imagery

Uses all seven project-confirmed parcels (486.6 acres). Run locally; outputs are saved in `goat_island_outputs/vegetation`.

Restores annual summer summaries for 2017–2026, common-valid-pixel comparisons for 2018/2022/2026, an NDVI change map, dated true-color/NDVI imagery, and interactive layers. Summer means June 1 through August 31 (September 1 exclusive) throughout.

Source: [Harmonized Sentinel-2 surface reflectance](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED). Early 2017–2018 coverage may be incomplete. Missing years stay missing; values are never interpolated. NDVI measures spectral greenness, not biodiversity or proven vegetation health.


Run the cells in order. Settings and helper functions are grouped at the top; calculations, exports, and visual review follow in separate sections. Figures are PNG only.

Dependencies (install once in your notebook environment if needed): `earthengine-api geemap geopandas requests matplotlib pandas shapely pyogrio rasterio pillow`.

## 1. Imports

Load the libraries used by this notebook.

In [ ]:
import hashlib
import json
import warnings
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
from IPython.display import display
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize, TwoSlopeNorm
from PIL import Image
from rasterio.features import geometry_mask
from rasterio.plot import show
from shapely import force_2d


## 2. Settings and local paths

Edit project settings here. Summer is June 1 through August 31. The same thresholds and comparison years are retained.

In [ ]:
EE_PROJECT = "stalwart-glider-279103"
SOURCE = "COPERNICUS/S2_SR_HARMONIZED"
ALL_YEARS = list(range(2017, 2027))
COMPARE_YEARS = [2018, 2022, 2026]
BAD_SCL = [0, 1, 3, 8, 9, 10, 11]
MIN_OBSERVATIONS = 1
METRIC_CRS = "EPSG:26914"
M_PER_MILE = 1609.344
ACRE_M2 = 4046.8564224
GOAT_ISLAND_PARCEL_IDS = [
    "65079661510030000",
    "65130905010020000",
    "65130905010030000",
    "65130905010050000",
    "65130905010060000",
    "65146334010100000",
    "65146334010140000",
]
NDVI_VIS = {"min": -0.2, "max": 0.9, "palette": ["ffffcc", "78c679", "006837"]}

PROJECT_ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "Park_Parcels/Park_Parcles.shp").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the goat-island-analysis folder.")
PARCEL_PATH = PROJECT_ROOT / "Park_Parcels/Park_Parcles.shp"
OUTPUT_DIR = PROJECT_ROOT / "goat_island_outputs"
VEG_DIR = OUTPUT_DIR / "vegetation"
VEG_DIR.mkdir(parents=True, exist_ok=True)


## 3. Earth Engine connection

Reuse existing authentication when available.

In [ ]:
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready. Vegetation outputs:", VEG_DIR)


## 4. Seven-parcel boundary and analysis grid

Select the confirmed parcel IDs, remove Z coordinates, and create the one-mile buffer. All NDVI summaries use a fixed 10 m UTM grid.

In [ ]:
parcels = gpd.read_file(PARCEL_PATH)
if parcels.crs is None:
    raise ValueError("Park parcels must have a defined coordinate system.")
parcel_numbers = parcels["Parcel_Num"].astype(str).str.strip()
missing_ids = set(GOAT_ISLAND_PARCEL_IDS) - set(parcel_numbers)
if missing_ids:
    raise ValueError(f"Required Goat Island parcels are missing: {sorted(missing_ids)}")
selected = parcels.loc[parcel_numbers.isin(GOAT_ISLAND_PARCEL_IDS)].copy()
if selected.empty:
    raise ValueError("No matching Goat Island project parcels.")
if selected.geometry.isna().any() or selected.geometry.is_empty.any():
    raise ValueError("Selected Goat Island parcels contain missing or empty geometry.")
selected["geometry"] = selected.geometry.map(force_2d).make_valid()
if not selected.geom_type.isin(["Polygon", "MultiPolygon"]).all():
    raise ValueError("Selected parcels must contain polygon geometry.")
preserve_gdf = selected.dissolve().reset_index(drop=True).to_crs("EPSG:4326")
BOUNDARY_SOURCE = (
    "Park_Parcels/Park_Parcles.shp; seven project-confirmed Parcel_Num values"
)
preserve_m = preserve_gdf.to_crs(METRIC_CRS)
study1_m = gpd.GeoDataFrame(geometry=preserve_m.buffer(M_PER_MILE), crs=METRIC_CRS)
preserve_area_acres = preserve_m.area.sum() / ACRE_M2
print(
    f"BOUNDARY: {BOUNDARY_SOURCE}; {len(selected)} parcel(s); area = {preserve_area_acres:.1f} acres"
)
preserve_gdf.to_file(OUTPUT_DIR / "preserve_boundary.geojson", driver="GeoJSON")
study1_m.to_crs(4326).to_file(
    OUTPUT_DIR / "one_mile_study_area.geojson", driver="GeoJSON"
)
roi = ee.Geometry(
    json.loads(json.dumps(preserve_gdf.geometry.iloc[0].__geo_interface__))
)
roi1 = ee.Geometry(
    json.loads(json.dumps(study1_m.to_crs(4326).geometry.iloc[0].__geo_interface__))
)


In [ ]:
xmin, ymin, xmax, ymax = preserve_m.total_bounds
GRID = [10, 0, float(np.floor(xmin / 10) * 10), 0, -10, float(np.ceil(ymax / 10) * 10)]
REDUCE = {
    "crs": METRIC_CRS,
    "crsTransform": GRID,
    "maxPixels": 100000000,
    "tileScale": 4,
}
BOUNDARY_HASH = hashlib.sha256(preserve_gdf.geometry.iloc[0].wkb).hexdigest()


## 5. Reusable helpers

The processing rules are defined once here. Yearly summaries remain sequential to avoid Earth Engine concurrent-aggregation limits.

In [ ]:
def summer_collection(year, geometry):
    """Find summer scenes intersecting a region, with the existing tile cloud threshold."""
    return (
        ee.ImageCollection(SOURCE)
        .filterBounds(geometry)
        .filterDate(f"{year}-06-01", f"{year}-09-01")
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 70))
    )


def clear_image(image):
    """Mask excluded SCL classes and retain the RGB/NIR bands used below."""
    scl = image.select("SCL")
    valid = scl.remap(BAD_SCL, [0] * len(BAD_SCL), 1).And(scl.mask())
    return image.updateMask(valid).select(["B2", "B3", "B4", "B8"])


def stats_feature(year, image, geometry):
    """Build mean, spatial standard deviation, and valid-area statistics for one year."""
    stats = image.reduceRegion(
        ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry,
        **REDUCE,
    )
    area = (
        ee.Image.pixelArea()
        .updateMask(image.mask())
        .rename("valid_area_m2")
        .reduceRegion(ee.Reducer.sum(), geometry, **REDUCE)
        .get("valid_area_m2")
    )
    return ee.Feature(None, stats).set(
        {"year": year, "scene_count": scene_counts.get(year), "valid_area_m2": area}
    )


def save_fig(fig, name):
    """Save a PNG figure and display it in the notebook."""
    fig.savefig(VEG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()


def export_raster(image, path, region=roi, grid=GRID):
    """Export an image on the fixed grid and mask pixels outside the matching study geometry."""
    if path.exists():
        path.unlink()
    geemap.ee_export_image(
        image.unmask(-9999),
        filename=str(path),
        region=region.bounds(),
        crs=METRIC_CRS,
        crs_transform=grid,
        file_per_band=False,
        timeout=180,
    )
    if not path.exists():
        raise RuntimeError(f"Raster export failed: {path}")
    local_geom = (
        preserve_m.geometry.iloc[0] if region is roi else study1_m.geometry.iloc[0]
    )
    with rasterio.open(path, "r+") as src:
        inside = geometry_mask(
            [local_geom.__geo_interface__],
            out_shape=src.shape,
            transform=src.transform,
            invert=True,
        )
        data = src.read()
        data[:, ~inside] = -9999
        src.write(data)
        src.nodata = -9999
    return path


def add_roi_invalid_pct(image):
    """Rank scene quality using cloud/invalid coverage over the seven parcels."""
    scl = image.select("SCL").unmask(0, False)
    invalid = scl.remap(BAD_SCL, [1] * len(BAD_SCL), 0).rename("invalid")
    fraction = invalid.reduceRegion(
        ee.Reducer.mean(), roi, crs=METRIC_CRS, scale=20, maxPixels=100000000
    ).get("invalid")
    return image.set("ROI_INVALID_PCT", ee.Number(fraction).multiply(100))


## 6. Build yearly summer composites

Mask clouds and build median-reflectance NDVI composites. Missing comparison years stop with a clear message.

In [ ]:
counts = ee.FeatureCollection(
    [
        ee.Feature(None, {"year": y, "scene_count": summer_collection(y, roi).size()})
        for y in ALL_YEARS
    ]
).getInfo()
scene_counts = {
    int(f["properties"]["year"]): int(f["properties"]["scene_count"])
    for f in counts["features"]
}
ndvi = {}
observations = {}
source_ids = {}
for year in ALL_YEARS:
    if not scene_counts[year]:
        continue
    collection = summer_collection(year, roi)
    masked = collection.map(clear_image)
    observations[year] = (
        masked.select("B8").count().rename("clear_observations").clip(roi)
    )
    composite = masked.median()
    ndvi[year] = (
        composite.normalizedDifference(["B8", "B4"])
        .rename("NDVI")
        .updateMask(observations[year].gte(MIN_OBSERVATIONS))
        .clip(roi)
    )
    source_ids[str(year)] = collection.aggregate_array("system:index").getInfo()
missing = set(COMPARE_YEARS) - set(ndvi)
if missing:
    raise RuntimeError(f"Comparison years have no scenes: {sorted(missing)}")


## 7. Summarize annual NDVI

Each year uses its own valid area. Missing years stay missing; coverage and scene counts are exported with the statistics.

In [ ]:
annual_rows = []
for year in ndvi:
    print("Summarizing summer", year, flush=True)
    annual_rows.append(stats_feature(year, ndvi[year], roi).getInfo()["properties"])
annual = (
    pd.DataFrame(annual_rows)
    .set_index("year")
    .reindex(ALL_YEARS)
    .rename_axis("year")
    .reset_index()
)
annual["scene_count"] = annual.year.map(scene_counts)
annual["valid_area_m2"] = annual.valid_area_m2.fillna(0)
annual["valid_area_acres"] = annual.valid_area_m2 / ACRE_M2
annual["valid_coverage_pct"] = 100 * annual.valid_area_acres / preserve_area_acres
annual["status"] = np.where(annual.NDVI_mean.notna(), "available", "no usable imagery")
annual.to_csv(VEG_DIR / "annual_ndvi_summary.csv", index=False)
display(annual)
if annual.valid_coverage_pct.lt(95).any():
    warnings.warn(
        "Some years have less than 95% coverage. Review coverage before interpreting differences."
    )


## 8. Compare the same valid pixels

2018, 2022, and 2026 use the intersection of their valid pixels. The change is 2026 minus 2018 in NDVI units.

In [ ]:
common_mask = (
    ee.ImageCollection.fromImages([ndvi[y].mask() for y in COMPARE_YEARS])
    .min()
    .rename("common_valid")
)
ndvi_common = {y: ndvi[y].updateMask(common_mask) for y in COMPARE_YEARS}
change = ndvi_common[2026].subtract(ndvi_common[2018]).rename("NDVI_change")
comparison = pd.DataFrame(
    [
        stats_feature(y, ndvi_common[y], roi).getInfo()["properties"]
        for y in COMPARE_YEARS
    ]
)
comparison["common_area_acres"] = comparison.valid_area_m2 / ACRE_M2
comparison["common_coverage_pct"] = (
    100 * comparison.common_area_acres / preserve_area_acres
)
if comparison.NDVI_mean.isna().any() or comparison.common_area_acres.le(0).any():
    raise RuntimeError("No common valid comparison area.")
comparison.to_csv(VEG_DIR / "selected_years_common_mask.csv", index=False)
change_stats = change.reduceRegion(
    ee.Reducer.mean().combine(ee.Reducer.minMax(), sharedInputs=True), roi, **REDUCE
).getInfo()
pd.DataFrame([change_stats]).to_csv(VEG_DIR / "ndvi_change_summary.csv", index=False)
display(comparison)
print("Change summary:", change_stats)


## 9. Plot the annual series and coverage

Selected-year common-mask means are shown separately from annual available-pixel means.

In [ ]:
fig, axes = plt.subplots(
    2, 1, figsize=(11, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
)
axes[0].plot(annual.year, annual.NDVI_mean, marker="o", label="Annual available pixels")
axes[0].plot(
    comparison.year,
    comparison.NDVI_mean,
    marker="s",
    linestyle="none",
    label="Selected years: common valid pixels",
)
for row in annual.itertuples():
    if pd.notna(row.NDVI_mean):
        axes[0].annotate(
            f"{row.NDVI_mean:.3f}",
            (row.year, row.NDVI_mean),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center",
            fontsize=8,
        )
axes[0].set(
    ylabel="Mean summer NDVI",
    title="Goat Island annual summer greenness — seven parcels",
)
axes[0].legend()
axes[0].grid(alpha=0.2)
axes[1].bar(annual.year, annual.valid_coverage_pct, color="gray")
axes[1].set(xlabel="Year", ylabel="Valid area (%)", ylim=(0, 105), xticks=ALL_YEARS)
fig.text(
    0.5,
    0.01,
    "Sentinel-2 SR Harmonized • June–August • 10 m grid • gaps indicate missing data; NDVI is not habitat quality",
    ha="center",
    fontsize=8,
)
fig.tight_layout(rect=[0, 0.05, 1, 1])
save_fig(fig, "annual_ndvi_trend")


## 10. Export analytical rasters

Save selected-year NDVI, the change layer, clear-observation counts, and the independent 2026 integration raster. NoData is -9999.

In [ ]:
cube = ee.Image.cat(
    [ndvi_common[y].rename(f"NDVI_{y}") for y in COMPARE_YEARS] + [change]
)
export_raster(cube, VEG_DIR / "selected_years_and_change_10m.tif")
export_raster(ndvi[2026], VEG_DIR / "ndvi_2026_10m.tif")
export_raster(
    ee.Image.cat(
        [observations[y].rename(f"observations_{y}") for y in COMPARE_YEARS]
    ).toFloat(),
    VEG_DIR / "clear_observations_10m.tif",
)


## 11. Plot selected years and NDVI change

All three NDVI maps share one color scale; the change scale is centered at zero.

In [ ]:
with rasterio.open(VEG_DIR / "selected_years_and_change_10m.tif") as src:
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    for i, ax in enumerate(axes.flat):
        values = src.read(i + 1, masked=True)
        if i < 3:
            show(
                values, transform=src.transform, ax=ax, cmap="YlGn", vmin=-0.2, vmax=0.9
            )
            ax.set_title(f"Summer {COMPARE_YEARS[i]} — common valid pixels")
            fig.colorbar(ax.images[0], ax=ax, shrink=0.7, label="NDVI")
        else:
            show(
                values,
                transform=src.transform,
                ax=ax,
                cmap="RdYlGn",
                norm=TwoSlopeNorm(vmin=-0.3, vcenter=0, vmax=0.3),
            )
            ax.set_title("NDVI change: 2026 minus 2018")
            fig.colorbar(
                ax.images[0], ax=ax, shrink=0.7, label="NDVI difference", extend="both"
            )
        preserve_m.boundary.plot(ax=ax, color="black", linewidth=0.7)
        ax.set(xlabel="UTM Easting (m)", ylabel="UTM Northing (m)")
        ax.ticklabel_format(style="plain", useOffset=False)
    fig.suptitle("Goat Island vegetation comparison — seven parcels")
    fig.tight_layout()
    save_fig(fig, "ndvi_comparison_and_change")


## 12. Select a dated clear scene

Rank June–August 2026 scenes by invalid coverage over the parcels. Calculate RGB contrast over the preview area once.

In [ ]:
ranked = (
    ee.ImageCollection(SOURCE)
    .filterBounds(roi)
    .filterDate("2026-06-01", "2026-09-01")
    .map(add_roi_invalid_pct)
    .sort("ROI_INVALID_PCT")
)
scene = ee.Image(ranked.first())
scene_date = scene.date().format("YYYY-MM-dd").getInfo()
invalid_pct = scene.get("ROI_INVALID_PCT").getInfo()
scene_clear = clear_image(scene)
recent_ndvi = scene_clear.normalizedDifference(["B8", "B4"]).rename("NDVI")
preview_geom = preserve_m.geometry.iloc[0].buffer(150).envelope
x0, y0, x1, y1 = preview_geom.bounds
preview_region = ee.Geometry.Rectangle(
    [x0, y0, x1, y1], proj=METRIC_CRS, geodesic=False
)
stretch = (
    scene_clear.select(["B4", "B3", "B2"])
    .reduceRegion(ee.Reducer.percentile([2, 98]), preview_region, **REDUCE)
    .getInfo()
)
minimum = [stretch.get(f"{b}_p2") for b in ["B4", "B3", "B2"]]
maximum = [stretch.get(f"{b}_p98") for b in ["B4", "B3", "B2"]]
if any((v is None for v in minimum + maximum)):
    raise RuntimeError("No clear RGB data for scene preview.")
maximum = [max(hi, lo + 1) for lo, hi in zip(minimum, maximum)]
rgb_vis = {"bands": ["B4", "B3", "B2"], "min": minimum, "max": maximum}
print(
    "Selected scene:",
    scene_date,
    "parcel invalid/cloud coverage:",
    round(invalid_pct, 2),
    "%",
)


## 13. Plot same-date RGB and NDVI

This scene supports visual interpretation; it does not replace summer-composite statistics.

In [ ]:
params = {
    "region": preview_region,
    "crs": METRIC_CRS,
    "dimensions": 1200,
    "format": "png",
}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, img, title in zip(
    axes,
    [
        scene_clear.visualize(**rgb_vis),
        recent_ndvi.visualize(
            min=-0.2, max=0.9, palette=["#ffffcc", "#78c679", "#006837"]
        ),
    ],
    ["True-color RGB (clear pixels)", "Same-date NDVI (clear pixels)"],
):
    response = requests.get(img.getThumbURL(params), timeout=120)
    response.raise_for_status()
    ax.imshow(
        Image.open(BytesIO(response.content)),
        extent=[x0, x1, y0, y1],
        interpolation="nearest",
    )
    preserve_m.boundary.plot(ax=ax, color="red", linewidth=0.8)
    ax.set(title=title, xlabel="UTM Easting (m)", ylabel="UTM Northing (m)")
    ax.ticklabel_format(style="plain", useOffset=False)
fig.suptitle(
    f"Goat Island Sentinel-2 — {scene_date}; parcel invalid/cloud pixels: {invalid_pct:.2f}%"
)
fig.subplots_adjust(right=0.89, wspace=0.22, top=0.82, bottom=0.17)
cax = fig.add_axes([0.91, 0.25, 0.012, 0.48])
fig.colorbar(
    ScalarMappable(
        norm=Normalize(-0.2, 0.9),
        cmap=LinearSegmentedColormap.from_list(
            "ndvi", ["#ffffcc", "#78c679", "#006837"]
        ),
    ),
    cax=cax,
    label="NDVI (right panel)",
)
fig.text(
    0.5,
    0.01,
    "ESA/Copernicus Sentinel-2 • 10 m RGB/NDVI; SCL cloud mask 20 m • red: seven-parcel boundary",
    ha="center",
    fontsize=8,
)
save_fig(fig, "dated_sentinel_rgb_ndvi")


## 14. Build the one-mile composite

Use scenes covering the full buffer; do not expand an already clipped parcel image.

In [ ]:
buffer_collection = summer_collection(2026, roi1).map(clear_image)
if buffer_collection.size().getInfo() == 0:
    raise RuntimeError("No summer scenes for the buffer.")
buffer_observations = buffer_collection.select("B8").count()
buffer_ndvi = (
    buffer_collection.median()
    .normalizedDifference(["B8", "B4"])
    .rename("NDVI")
    .updateMask(buffer_observations.gte(MIN_OBSERVATIONS))
    .clip(roi1)
)
export_raster(buffer_ndvi, VEG_DIR / "ndvi_2026_one_mile_10m.tif", region=roi1)


## 15. Plot the surroundings

Keep this contextual map distinct from parcel-only summaries.

In [ ]:
with rasterio.open(VEG_DIR / "ndvi_2026_one_mile_10m.tif") as src:
    fig, ax = plt.subplots(figsize=(10, 8))
    show(
        src.read(1, masked=True),
        transform=src.transform,
        ax=ax,
        cmap="YlGn",
        vmin=-0.2,
        vmax=0.9,
    )
    preserve_m.boundary.plot(ax=ax, color="black", linewidth=1)
    study1_m.boundary.plot(ax=ax, color="gray", linestyle="--", linewidth=1)
    fig.colorbar(ax.images[0], ax=ax, label="Summer composite NDVI")
    ax.set(
        title="Goat Island and one-mile surroundings — summer 2026",
        xlabel="UTM Easting (m)",
        ylabel="UTM Northing (m)",
    )
    ax.ticklabel_format(style="plain", useOffset=False)
    save_fig(fig, "ndvi_2026_one_mile_map")


## 16. Record provenance

Record the boundary, data source, image IDs, masking rules, and selected scene for reproducibility.

In [ ]:
manifest = {
    "boundary_hash": BOUNDARY_HASH,
    "parcel_ids": sorted(GOAT_ISLAND_PARCEL_IDS),
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "source": SOURCE,
    "summer_window": "June 1 to September 1 exclusive",
    "tile_cloud_threshold_pct": 70,
    "bad_scl": BAD_SCL,
    "minimum_clear_observations": MIN_OBSERVATIONS,
    "analysis_crs": METRIC_CRS,
    "grid_transform": GRID,
    "annual_series_mask": "year-specific valid pixels; missing years retained as gaps",
    "comparison_years": COMPARE_YEARS,
    "comparison_mask": "intersection of valid pixels in all comparison years",
    "selected_scene": {
        "id": scene.get("system:index").getInfo(),
        "date": scene_date,
        "parcel_invalid_pct": invalid_pct,
    },
    "source_image_ids": source_ids,
    "integration_raster": "ndvi_2026_10m.tif",
}
(VEG_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")


## 17. Interactive review

Toggle historical composites, change, dated imagery, buffer NDVI, and the boundary outlines.

In [ ]:
m = geemap.Map()
m.centerObject(roi1, 13)
m.addLayer(scene_clear, rgb_vis, f"Sentinel RGB {scene_date}", False)
m.addLayer(recent_ndvi, NDVI_VIS, f"Single-scene NDVI {scene_date}", False)
for year in COMPARE_YEARS:
    m.addLayer(
        ndvi_common[year], NDVI_VIS, f"Common-mask summer NDVI {year}", year == 2026
    )
m.addLayer(
    change,
    {"min": -0.3, "max": 0.3, "palette": ["d73027", "ffffbf", "1a9850"]},
    "NDVI change 2026 minus 2018",
    False,
)
m.addLayer(buffer_ndvi, NDVI_VIS, "2026 buffer NDVI", False)
m.addLayer(
    ee.FeatureCollection([ee.Feature(roi)]).style(
        color="ff0000", fillColor="00000000", width=2
    ),
    {},
    "Seven selected parcels",
)
m.addLayer(
    ee.FeatureCollection([ee.Feature(roi1)]).style(
        color="555555", fillColor="00000000", width=1
    ),
    {},
    "One-mile boundary",
)
m.add_colorbar(NDVI_VIS, label="NDVI (change layer has a separate −0.3 to +0.3 scale)")
display(m)
print("Outputs:", VEG_DIR)


## Interpretation and review
- Inspect annual coverage and clear-observation rasters; scene count is not the number of valid observations at every pixel.
- Compare selected-year means from `selected_years_common_mask.csv` when making like-for-like statements. Do not mix them with the year-specific annual means without explaining the mask difference.
- Change is an NDVI difference between summer composites. It does not establish a cause or a statistically significant ecological trend.
- The single scene uses the same June–August window as the composites and is for visual inspection.
- All results refer to the seven specified parcels. Use the buffer output only when discussing the surrounding study area.